In [ ]:
!pip install geopandas plotly pandas geopy


**Wildfire Dataset - AK Fire Points**

Name: AK Fire Points

Temporal Coverage: Fires from 1939 - 2024


**Variables:**

NAME - The name assigned to an incident.

FIRESEASON - Year the fire started.

PRESCRIBEDFIRE - This field describes whether the fire type was a prescribed fire (i.e., planned ignition). (Y or N)

LATITUDE - Latitude coordinate of where the incident started. Format is decimal degrees.

LONGITUDE - Longitude coordinate of where the incident started. Format is decimal degrees.

DISCOVERYDATETIME - Date and time the fire was discovered.

DISCOVERYSIZE - Estimated size of fire at time of discovery, expressed in acres.

OUTDATE - Date the fire was classified as out.

ESTIMATEDTOTALACRES - Estimated total acres of fire.

SPECIFICCAUSE - Specific cause of the fire, if known.

ORIGINMERIDIAN - Public Land Survey Meridian of where the fire started.


PRIMARYFUELTYPE - Primary fuel type, if known.

Format: CSV file





In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.express as px
import gdown
import plotly.graph_objects as go
from google.colab import drive

#AK fire location points dataset in csv file format
gdown.download(
    'https://drive.google.com/uc?id=1frpPRr-c8YNkdVtLadeRPU5rrw3Xbmo_',
    'AK_fire_location_points.csv',
    quiet=False
)

df = pd.read_csv('AK_fire_location_points.csv',
                 usecols=['NAME', 'FIRESEASON', 'PRESCRIBEDFIRE', 'LATITUDE', 'LONGITUDE', 'DISCOVERYDATETIME', 'DISCOVERYSIZE',
                          'OUTDATE', 'ESTIMATEDTOTALACRES', 'SPECIFICCAUSE', 'ORIGINMERIDIAN', 'PRIMARYFUELTYPE'])

#Drop missing fields
df = df.dropna(subset=["LATITUDE","LONGITUDE","FIRESEASON"])
df["FIRESEASON"] = pd.to_numeric(df["FIRESEASON"], errors="coerce")

#Drop prescribed fires
df = df[df['PRESCRIBEDFIRE'] == 'N']

In [ ]:
lat_min, lat_max = 54.0, 72.0
lon_min, lon_max = -170.0, -130.0

df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')

df_alaska = df[
    (df['LATITUDE'] >= lat_min) & (df['LATITUDE'] <= lat_max) &
    (df['LONGITUDE'] >= lon_min) & (df['LONGITUDE'] <= lon_max)
].copy()

df_alaska['Year'] = (
    pd.to_datetime(df_alaska['DISCOVERYDATETIME'], errors='coerce')
    .dt.year
)
df_alaska = df_alaska.dropna(subset=['Year'])
df_alaska['Year'] = df_alaska['Year'].astype(int)

df_alaska = df_alaska[df_alaska['Year'] <= 2024].copy()

df_alaska = df_alaska.sort_values('Year')

df_alaska.dropna(subset=['ESTIMATEDTOTALACRES'], inplace=True)

# Fill empty 'SPECIFICCAUSE' with 'Unknown'
df_alaska['SPECIFICCAUSE'] = df_alaska['SPECIFICCAUSE'].fillna('Unknown')

df_alaska['DISCOVERYDATE'] = pd.to_datetime(df_alaska['DISCOVERYDATETIME'], errors='coerce').dt.date

bins = [0, 100, 1000, 10000, df_alaska['ESTIMATEDTOTALACRES'].max() + 1]
labels = ['< 100 acres', '100 - 1,000 acres', '1,000 - 10,000 acres', '> 10,000 acres']

category_type = pd.CategoricalDtype(categories=labels, ordered=True)
df_alaska['Fire_Size_Category'] = pd.cut(df_alaska['ESTIMATEDTOTALACRES'], bins=bins, labels=labels, right=False).astype(category_type)

category_marker_sizes = {
    '< 100 acres': 6,
    '100 - 1,000 acres': 12,
    '1,000 - 10,000 acres': 20,
    '> 10,000 acres': 30
}

df_alaska['Marker_Size'] = df_alaska['Fire_Size_Category'].map(category_marker_sizes)

category_colors = {
    '< 100 acres': 'lightgreen',
    '100 - 1,000 acres': 'yellow',
    '1,000 - 10,000 acres': 'orange',
    '> 10,000 acres': 'red'
}

df_alaska.rename(columns={
    'NAME': 'Name',
    'LATITUDE': 'Latitude',
    'LONGITUDE': 'Longitude',
    'DISCOVERYDATE': 'DiscoveryDate',
    'ESTIMATEDTOTALACRES': 'EstimatedTotalAcres',
    'SPECIFICCAUSE': 'SpecificCause',
    'Year': 'Year'
}, inplace=True)

fig = px.scatter_mapbox(
    df_alaska,
    lat='Latitude',
    lon='Longitude',
    color='Fire_Size_Category',
    color_discrete_map=category_colors,
    size='Marker_Size',
    category_orders={'Fire_Size_Category': labels},
    hover_name='Name',
    hover_data=['Year', 'Latitude', 'Longitude',
                'DiscoveryDate', 'EstimatedTotalAcres', 'SpecificCause'],
    mapbox_style='carto-positron',
    zoom=3.5,
    center={"lat": 63.0, "lon": -150.0},
    animation_frame='Year',
    title='AK Fire Locations by Estimated Total Acres Burned (Animated by Year)'
)

fig.update_layout(height=1000, width=1000)
fig.show()